In [5]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import MobileNetV3Small
from sklearn.metrics import classification_report, confusion_matrix

DATASET_DIR = "plantvillage"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_PHASE1 = 10
EPOCHS_PHASE2 = 10
LEARNING_RATE_PHASE1 = 1e-3
LEARNING_RATE_PHASE2 = 1e-5
VALIDATION_SPLIT = 0.2
SEED = 42
MODEL_EXPORT_DIR = "models/disease_detection"

os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")
print(f"Metal GPU: {'Yes' if tf.config.list_physical_devices('GPU') else 'No (CPU mode)'}")

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
dataset_path = Path(DATASET_DIR)
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at '{DATASET_DIR}'. "
        f"Please download PlantVillage dataset and extract the 'color' folder here."
    )

class_dirs = sorted([d for d in dataset_path.iterdir() if d.is_dir()])
class_names = [d.name for d in class_dirs]
class_counts = {d.name: len(list(d.glob('*.[jJpP][pPnN][gG]'))) for d in class_dirs}

print(f"Number of classes: {len(class_names)}")
print(f"Total images: {sum(class_counts.values())}")
for name, count in class_counts.items():
    status = "healthy" if "healthy" in name.lower() else "disease"
    print(f"  {name}: {count} images ({status})")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
colors = ['#4CAF50' if 'healthy' in name.lower() else '#F44336' for name in class_counts.keys()]
ax.barh(list(class_counts.keys()), list(class_counts.values()), color=colors)
ax.set_xlabel('Number of Images')
ax.set_title('PlantVillage Dataset — Class Distribution')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{MODEL_EXPORT_DIR}/class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: class_distribution.png")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
rng = np.random.default_rng(SEED)
sample_classes = rng.choice(class_dirs, size=8, replace=False)

for ax, cls_dir in zip(axes.flat, sample_classes):
    images = list(cls_dir.glob('*.[jJpP][pPnN][gG]'))
    img_path = rng.choice(images)
    img = keras.utils.load_img(img_path, target_size=IMG_SIZE)
    ax.imshow(img)
    label = cls_dir.name.replace('___', '\n').replace('_', ' ')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Images from PlantVillage Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{MODEL_EXPORT_DIR}/sample_images.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

val_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Classes loaded: {num_classes}")
print(f"Training batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}")

In [ ]:
class_mapping = {i: name for i, name in enumerate(class_names)}
with open(f"{MODEL_EXPORT_DIR}/class_names.json", "w") as f:
    json.dump(class_mapping, f, indent=2)
print(f"Saved class mapping ({num_classes} classes) to class_names.json")

treatments = {}
for name in class_names:
    if 'healthy' in name.lower():
        treatments[name] = {
            "en": "No treatment needed. Plant appears healthy. Continue regular care and monitoring.",
            "ar": "\u0644\u0627 \u064a\u0644\u0632\u0645 \u0639\u0644\u0627\u062c. \u0627\u0644\u0646\u0628\u0627\u062a \u064a\u0628\u062f\u0648 \u0628\u0635\u062d\u0629 \u062c\u064a\u062f\u0629. \u0627\u0633\u062a\u0645\u0631 \u0641\u064a \u0627\u0644\u0631\u0639\u0627\u064a\u0629 \u0648\u0627\u0644\u0645\u0631\u0627\u0642\u0628\u0629 \u0627\u0644\u0645\u0646\u062a\u0638\u0645\u0629."
        }
    elif 'scab' in name.lower():
        treatments[name] = {
            "en": "Apply fungicide spray. Remove infected leaves. Improve air circulation around plants.",
            "ar": "\u0631\u0634 \u0645\u0628\u064a\u062f \u0641\u0637\u0631\u064a. \u0625\u0632\u0627\u0644\u0629 \u0627\u0644\u0623\u0648\u0631\u0627\u0642 \u0627\u0644\u0645\u0635\u0627\u0628\u0629. \u062a\u062d\u0633\u064a\u0646 \u062f\u0648\u0631\u0627\u0646 \u0627\u0644\u0647\u0648\u0627\u0621 \u062d\u0648\u0644 \u0627\u0644\u0646\u0628\u0627\u062a\u0627\u062a."
        }
    elif 'rot' in name.lower():
        treatments[name] = {
            "en": "Remove and destroy infected fruit. Apply copper-based fungicide. Prune to improve airflow.",
            "ar": "\u0625\u0632\u0627\u0644\u0629 \u0648\u062a\u062f\u0645\u064a\u0631 \u0627\u0644\u062b\u0645\u0627\u0631 \u0627\u0644\u0645\u0635\u0627\u0628\u0629. \u0631\u0634 \u0645\u0628\u064a\u062f \u0641\u0637\u0631\u064a \u0646\u062d\u0627\u0633\u064a. \u062a\u0642\u0644\u064a\u0645 \u0644\u062a\u062d\u0633\u064a\u0646 \u062a\u062f\u0641\u0642 \u0627\u0644\u0647\u0648\u0627\u0621."
        }
    elif 'blight' in name.lower():
        treatments[name] = {
            "en": "Apply fungicide immediately. Remove severely affected plants. Avoid overhead watering.",
            "ar": "\u0631\u0634 \u0645\u0628\u064a\u062f \u0641\u0637\u0631\u064a \u0641\u0648\u0631\u0627\u064b. \u0625\u0632\u0627\u0644\u0629 \u0627\u0644\u0646\u0628\u0627\u062a\u0627\u062a \u0627\u0644\u0645\u062a\u0636\u0631\u0631\u0629 \u0628\u0634\u062f\u0629. \u062a\u062c\u0646\u0628 \u0627\u0644\u0631\u064a \u0627\u0644\u0639\u0644\u0648\u064a."
        }
    elif 'mold' in name.lower() or 'mildew' in name.lower():
        treatments[name] = {
            "en": "Improve ventilation. Reduce humidity. Apply sulfur-based fungicide.",
            "ar": "\u062a\u062d\u0633\u064a\u0646 \u0627\u0644\u062a\u0647\u0648\u064a\u0629. \u062a\u0642\u0644\u064a\u0644 \u0627\u0644\u0631\u0637\u0648\u0628\u0629. \u0631\u0634 \u0645\u0628\u064a\u062f \u0641\u0637\u0631\u064a \u0623\u0633\u0627\u0633\u0647 \u0627\u0644\u0643\u0628\u0631\u064a\u062a."
        }
    elif 'virus' in name.lower() or 'mosaic' in name.lower() or 'curl' in name.lower():
        treatments[name] = {
            "en": "Remove infected plants to prevent spread. Control insect vectors. No chemical cure for viral infections.",
            "ar": "\u0625\u0632\u0627\u0644\u0629 \u0627\u0644\u0646\u0628\u0627\u062a\u0627\u062a \u0627\u0644\u0645\u0635\u0627\u0628\u0629 \u0644\u0645\u0646\u0639 \u0627\u0644\u0627\u0646\u062a\u0634\u0627\u0631. \u0645\u0643\u0627\u0641\u062d\u0629 \u0627\u0644\u062d\u0634\u0631\u0627\u062a \u0627\u0644\u0646\u0627\u0642\u0644\u0629. \u0644\u0627 \u064a\u0648\u062c\u062f \u0639\u0644\u0627\u062c \u0643\u064a\u0645\u064a\u0627\u0626\u064a \u0644\u0644\u0639\u062f\u0648\u0649 \u0627\u0644\u0641\u064a\u0631\u0648\u0633\u064a\u0629."
        }
    elif 'rust' in name.lower():
        treatments[name] = {
            "en": "Remove infected leaves. Apply fungicide. Avoid wetting foliage.",
            "ar": "\u0625\u0632\u0627\u0644\u0629 \u0627\u0644\u0623\u0648\u0631\u0627\u0642 \u0627\u0644\u0645\u0635\u0627\u0628\u0629. \u0631\u0634 \u0645\u0628\u064a\u062f \u0641\u0637\u0631\u064a. \u062a\u062c\u0646\u0628 \u062a\u0628\u0644\u064a\u0644 \u0627\u0644\u0623\u0648\u0631\u0627\u0642."
        }
    elif 'bacterial' in name.lower():
        treatments[name] = {
            "en": "Remove infected parts. Apply copper-based bactericide. Avoid working with wet plants.",
            "ar": "\u0625\u0632\u0627\u0644\u0629 \u0627\u0644\u0623\u062c\u0632\u0627\u0621 \u0627\u0644\u0645\u0635\u0627\u0628\u0629. \u0631\u0634 \u0645\u0628\u064a\u062f \u0628\u0643\u062a\u064a\u0631\u064a \u0646\u062d\u0627\u0633\u064a. \u062a\u062c\u0646\u0628 \u0627\u0644\u0639\u0645\u0644 \u0645\u0639 \u0627\u0644\u0646\u0628\u0627\u062a\u0627\u062a \u0627\u0644\u0645\u0628\u0644\u0644\u0629."
        }
    else:
        treatments[name] = {
            "en": "Consult a local agricultural expert. Remove severely affected parts. Monitor spread closely.",
            "ar": "\u0627\u0633\u062a\u0634\u0631 \u062e\u0628\u064a\u0631\u0627\u064b \u0632\u0631\u0627\u0639\u064a\u0627\u064b \u0645\u062d\u0644\u064a\u0627\u064b. \u0625\u0632\u0627\u0644\u0629 \u0627\u0644\u0623\u062c\u0632\u0627\u0621 \u0627\u0644\u0645\u062a\u0636\u0631\u0631\u0629 \u0628\u0634\u062f\u0629. \u0645\u0631\u0627\u0642\u0628\u0629 \u0627\u0644\u0627\u0646\u062a\u0634\u0627\u0631 \u0639\u0646 \u0643\u062b\u0628."
        }

with open(f"{MODEL_EXPORT_DIR}/treatments.json", "w", encoding="utf-8") as f:
    json.dump(treatments, f, indent=2, ensure_ascii=False)
print(f"Saved treatment recommendations for {len(treatments)} classes")

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="data_augmentation")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Data pipeline ready")
print("Augmentation: RandomFlip, RandomRotation(0.2), RandomZoom(0.15), RandomContrast(0.1)")

In [ ]:
base_model = MobileNetV3Small(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=True
)
base_model.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs, name="zeraatech_disease_detector")

print(f"Model: {model.name}")
print(f"Classes: {num_classes}")
print(f"Total parameters: {model.count_params():,}")
trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f"Trainable parameters: {trainable:,}")

In [ ]:
model.summary(show_trainable=True)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_PHASE1),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

phase1_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

print("=" * 60)
print("PHASE 1: Training classification head (base model frozen)")
print(f"Epochs: {EPOCHS_PHASE1} | LR: {LEARNING_RATE_PHASE1} | Batch: {BATCH_SIZE}")
print("=" * 60)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=phase1_callbacks
)

phase1_val_acc = max(history_phase1.history['val_accuracy'])
print(f"\nPhase 1 complete — Best val accuracy: {phase1_val_acc:.4f} ({phase1_val_acc*100:.2f}%)")

In [ ]:
base_model.trainable = True
total_layers = len(base_model.layers)
fine_tune_from = int(total_layers * 0.7)

for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_PHASE2),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

phase2_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    callbacks.ModelCheckpoint(
        f"{MODEL_EXPORT_DIR}/best_model.keras",
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

print("=" * 60)
print("PHASE 2: Fine-tuning top layers of MobileNetV3")
print(f"Epochs: {EPOCHS_PHASE2} | LR: {LEARNING_RATE_PHASE2} | Batch: {BATCH_SIZE}")
print("=" * 60)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=phase2_callbacks
)

phase2_val_acc = max(history_phase2.history['val_accuracy'])
print(f"\nPhase 2 complete — Best val accuracy: {phase2_val_acc:.4f} ({phase2_val_acc*100:.2f}%)")

In [ ]:
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']
phase1_epochs = len(history_phase1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(acc, label='Train Accuracy', color='#2196F3', linewidth=2)
ax1.plot(val_acc, label='Val Accuracy', color='#4CAF50', linewidth=2)
ax1.axvline(x=phase1_epochs - 0.5, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
ax1.axhline(y=0.9, color='red', linestyle=':', alpha=0.5, label='SDD Target (90%)')
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0.5, 1.0])

ax2.plot(loss, label='Train Loss', color='#F44336', linewidth=2)
ax2.plot(val_loss, label='Val Loss', color='#FF9800', linewidth=2)
ax2.axvline(x=phase1_epochs - 0.5, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.suptitle('ZeraaTech Disease Detection — Training History', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{MODEL_EXPORT_DIR}/training_history.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_history.png")

In [ ]:
print("Evaluating model on validation set...")

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

overall_acc = np.mean(y_true == y_pred)
print(f"\n{'='*60}")
print(f"OVERALL VALIDATION ACCURACY: {overall_acc*100:.2f}%")
print(f"SDD TARGET: >=90%")
print(f"STATUS: {'TARGET MET' if overall_acc >= 0.9 else 'BELOW TARGET'}")
print(f"{'='*60}")

In [ ]:
display_names = [n.replace('___', ' - ').replace('_', ' ') for n in class_names]

report_text = classification_report(y_true, y_pred, target_names=display_names)
print("Classification Report:")
print(report_text)

report_dict = classification_report(y_true, y_pred, target_names=display_names, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(f"{MODEL_EXPORT_DIR}/classification_report.csv")
print(f"Saved: classification_report.csv")

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=display_names, yticklabels=display_names,
    ax=ax, linewidths=0.5, linecolor='lightgray'
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (Accuracy: {overall_acc*100:.2f}%)', fontsize=14, fontweight='bold')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(f"{MODEL_EXPORT_DIR}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)
class_acc_df = pd.DataFrame({'class': display_names, 'accuracy': per_class_acc}).sort_values('accuracy')

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#F44336' if acc < 0.9 else '#4CAF50' for acc in class_acc_df['accuracy']]
ax.barh(class_acc_df['class'], class_acc_df['accuracy'], color=colors)
ax.axvline(x=0.9, color='red', linestyle='--', alpha=0.7, label='90% target')
ax.set_xlabel('Accuracy')
ax.set_title('Per-Class Accuracy (sorted)', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1.05])
ax.legend()
plt.tight_layout()
plt.savefig(f"{MODEL_EXPORT_DIR}/per_class_accuracy.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
model.save(f"{MODEL_EXPORT_DIR}/disease_model.keras")
print(f"Saved Keras model: {MODEL_EXPORT_DIR}/disease_model.keras")

model.export(f"{MODEL_EXPORT_DIR}/saved_model")
print(f"Saved TF SavedModel: {MODEL_EXPORT_DIR}/saved_model/")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = f"{MODEL_EXPORT_DIR}/disease_model.tflite"
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

keras_size = os.path.getsize(f"{MODEL_EXPORT_DIR}/disease_model.keras") / (1024 * 1024)
tflite_size = os.path.getsize(tflite_path) / (1024 * 1024)

print(f"\nModel Sizes:")
print(f"  Keras model:  {keras_size:.1f} MB")
print(f"  TFLite model: {tflite_size:.1f} MB")
print(f"  Size reduction: {(1 - tflite_size/keras_size)*100:.0f}%")

In [ ]:
import time

loaded_model = keras.models.load_model(f"{MODEL_EXPORT_DIR}/disease_model.keras")

with open(f"{MODEL_EXPORT_DIR}/class_names.json") as f:
    class_map = json.load(f)
with open(f"{MODEL_EXPORT_DIR}/treatments.json") as f:
    treatment_map = json.load(f)

def predict_disease(image_path, mdl=loaded_model):
    start = time.time()
    img = keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = np.expand_dims(keras.utils.img_to_array(img), axis=0)
    preds = mdl.predict(img_array, verbose=0)
    inference_time = time.time() - start
    top3_idx = np.argsort(preds[0])[-3:][::-1]
    results = []
    for idx in top3_idx:
        name = class_map[str(idx)]
        treatment = treatment_map.get(name, {})
        results.append({
            "class": name,
            "confidence": float(preds[0][idx]),
            "treatment_en": treatment.get("en", ""),
            "treatment_ar": treatment.get("ar", "")
        })
    return {"prediction": results[0], "top3": results, "inference_time_ms": round(inference_time * 1000, 1)}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
rng = np.random.default_rng(42)
sample_classes = rng.choice(class_dirs, size=6, replace=False)

for ax, cls_dir in zip(axes.flat, sample_classes):
    images = list(cls_dir.glob('*.[jJpP][pPnN][gG]'))
    img_path = str(rng.choice(images))
    result = predict_disease(img_path)
    pred = result['prediction']
    img = keras.utils.load_img(img_path, target_size=IMG_SIZE)
    ax.imshow(img)
    predicted = pred['class'].replace('___', ' - ').replace('_', ' ')
    title = f"{predicted}\n{pred['confidence']*100:.1f}% | {result['inference_time_ms']}ms"
    color = 'green' if pred['class'] == cls_dir.name else 'red'
    ax.set_title(title, fontsize=8, color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('Inference Test — Sample Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{MODEL_EXPORT_DIR}/inference_test.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
summary = {
    "model_name": "ZeraaTech Disease Detector v1.0",
    "base_architecture": "MobileNetV3-Small",
    "training_strategy": "Transfer Learning (2-phase)",
    "dataset": "PlantVillage",
    "total_images": sum(class_counts.values()),
    "num_classes": num_classes,
    "validation_split": VALIDATION_SPLIT,
    "image_size": f"{IMG_SIZE[0]}x{IMG_SIZE[1]}",
    "phase1_epochs": len(history_phase1.history['accuracy']),
    "phase2_epochs": len(history_phase2.history['accuracy']),
    "final_val_accuracy": round(overall_acc * 100, 2),
    "sdd_target_met": overall_acc >= 0.9,
    "keras_model_size_mb": round(keras_size, 1),
    "tflite_model_size_mb": round(tflite_size, 1),
    "framework": f"TensorFlow {tf.__version__}",
    "training_date": datetime.now().strftime("%Y-%m-%d")
}

with open(f"{MODEL_EXPORT_DIR}/model_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 60)
print("     ZERAATECH DISEASE DETECTION — MODEL CARD")
print("=" * 60)
for key, value in summary.items():
    print(f"  {key.replace('_', ' ').title():.<40} {value}")
print("=" * 60)
print(f"\nAll files exported to: {MODEL_EXPORT_DIR}/")
print("  disease_model.keras  -> Flask service")
print("  disease_model.tflite -> Edge deployment")
print("  class_names.json     -> Label mapping")
print("  treatments.json      -> Treatment advice EN/AR")